<h1>Table of Contents<span class="tocSkip"></span></h1>
<div class="toc"><ul class="toc-item"><li><span><a href="#Сокращенная-предобработка" data-toc-modified-id="Сокращенная-предобработка-1"><span class="toc-item-num">1&nbsp;&nbsp;</span>Сокращенная предобработка</a></span></li><li><span><a href="#Нейронная-сеть" data-toc-modified-id="Нейронная-сеть-2"><span class="toc-item-num">2&nbsp;&nbsp;</span>Нейронная сеть</a></span><ul class="toc-item"><li><span><a href="#Задание-слоев-нейронной-сети-и-компиляция-модели" data-toc-modified-id="Задание-слоев-нейронной-сети-и-компиляция-модели-2.1"><span class="toc-item-num">2.1&nbsp;&nbsp;</span>Задание слоев нейронной сети и компиляция модели</a></span></li><li><span><a href="#Обучение-и-оценка-модели" data-toc-modified-id="Обучение-и-оценка-модели-2.2"><span class="toc-item-num">2.2&nbsp;&nbsp;</span>Обучение и оценка модели</a></span></li></ul></li></ul></div>

In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, ConfusionMatrixDisplay

In [16]:
file_path = "synthetic_dataset.xlsx"
df = pd.read_excel(file_path)

In [17]:
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [18]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

## Сокращенная предобработка

In [19]:
df.head()

,ID,Name,Age,Salary,City,Purchase_Amount,Total_Purchases,Days_Since_Last_Purchase,Signup_Date,Participated_In_Promo
0,1,Grace,54.0,110955.0,New York,455.0,1,185,2020-07-01,0
1,2,David,35.0,35433.0,London,NaN,8,217,2020-07-10,1
2,3,Hannah,45.0,88011.0,Berlin,158.0,4,315,2020-03-05,0
3,4,Eve,30.0,106688.0,Berlin,125.0,9,216,2022-08-20,0
4,5,Grace,NaN,82468.0,New York,NaN,5,119,2022-05-13,0


In [20]:
df['Participated_In_Promo'].value_counts(normalize=True)

Participated_In_Promo
0    0.779048
1    0.220952
Name: proportion, dtype: float64

In [21]:
# Заполняем числовые пропуски медианой (устойчиво к выбросам)
df['Age'].fillna(df['Age'].median(), inplace=True)

# Заполняем категориальные пропуски модой (наиболее частая категория)
df['City'].fillna(df['City'].mode()[0], inplace=True)

# Заполняем пропуски в сумме покупок средним значением
df['Purchase_Amount'].fillna(df['Purchase_Amount'].mean(), inplace=True)

In [22]:
# Если переменная разная для разных категорий, можно заполнять средним внутри каждой категории

df['Salary'] = df.groupby('City')['Salary'].transform(lambda x: x.fillna(x.mean()))

In [23]:
# Функция для удаления выбросов по IQR
def remove_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return df[(df[column] >= lower_bound) & (df[column] <= upper_bound)]

# Применяем обработку выбросов
df = remove_outliers(df, 'Age')
df = remove_outliers(df, 'Salary')

In [24]:
# Удаляем полные дубликаты
df = df.drop_duplicates()

In [25]:
## Биннинг числовых признаков (группировка по диапазонам)
df['Age_Group'] = pd.cut(df['Age'], bins=[0, 25, 45, 65, 100], labels=['Young', 'Adult', 'Middle_Aged', 'Senior'])
df['Salary_Group'] = pd.qcut(df['Salary'], q=4, labels=['Low', 'Medium', 'High', 'Very_High'])

In [26]:
## Извлечение информации из дат
df['Signup_Year'] = df['Signup_Date'].dt.year
df['Signup_Month'] = df['Signup_Date'].dt.month
df['Signup_Weekday'] = df['Signup_Date'].dt.weekday  # День недели (0 - Пн, 6 - Вс)
df['Signup_Weekend'] = (df['Signup_Weekday'] >= 5).astype(int)  # Флаг выходного дня

In [27]:
## Создание взаимодействий между признаками
df['Purchase_Frequency'] = df['Total_Purchases'] / (df['Days_Since_Last_Purchase'] + 1) * 100  # Частота покупок
df['Salary_to_Age'] = df['Salary'] / df['Age']   # Отношение зарплаты к возрасту
df['Spending_Score'] = df['Purchase_Amount'] / df['Salary'] * 100  # Какую часть зарплаты тратит пользователь

In [28]:
## Логические признаки
df['High_Income'] = (df['Salary'] > df['Salary'].median()).astype(int)  # Флаг высокой зарплаты
df['Loyal_Customer'] = (df['Total_Purchases'] > df['Total_Purchases'].median()).astype(int)  # Лояльный покупатель

In [29]:
category_features = ['City', 'Age_Group', 'Salary_Group', 'Signup_Year', 'Signup_Month', 'Signup_Weekday']

In [30]:
# 2. One-Hot Encoding (для линейных моделей) - удаляет исходный столбец
df_one_hot_enc = pd.get_dummies(df, columns=category_features, drop_first=True)

In [31]:
df_one_hot_enc.head()

,ID,Name,Age,Salary,Purchase_Amount,Total_Purchases,Days_Since_Last_Purchase,Signup_Date,Participated_In_Promo,Signup_Weekend,Purchase_Frequency,Salary_to_Age,Spending_Score,High_Income,Loyal_Customer,City_London,City_New York,City_Paris,Age_Group_Adult,Age_Group_Middle_Aged,Age_Group_Senior,Salary_Group_Medium,Salary_Group_High,Salary_Group_Very_High,Signup_Year_2021,Signup_Year_2022,Signup_Year_2023,Signup_Month_2,Signup_Month_3,Signup_Month_4,Signup_Month_5,Signup_Month_6,Signup_Month_7,Signup_Month_8,Signup_Month_9,Signup_Month_10,Signup_Month_11,Signup_Month_12,Signup_Weekday_1,Signup_Weekday_2,Signup_Weekday_3,Signup_Weekday_4,Signup_Weekday_5,Signup_Weekday_6
0,1,Grace,54.0,110955.0,455.000000,1,185,2020-07-01,0,0,0.537634,2054.722222,0.410076,1,0,False,True,False,False,True,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False
1,2,David,35.0,35433.0,280.267516,8,217,2020-07-10,1,0,3.669725,1012.371429,0.790979,0,1,True,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False
2,3,Hannah,45.0,88011.0,158.000000,4,315,2020-03-05,0,0,1.265823,1955.800000,0.179523,1,0,False,False,False,True,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False
3,4,Eve,30.0,106688.0,125.000000,9,216,2022-08-20,0,1,4.147465,3556.266667,0.117164,1,1,False,False,False,True,False,False,False,True,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False
4,5,Grace,34.0,82468.0,280.267516,5,119,2022-05-13,0,0,4.166667,2425.529412,0.339850,0,0,False,True,False,True,False,False,True,False,False,False,True,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False


In [32]:
X = df_one_hot_enc.drop(['Participated_In_Promo', 'ID', 'Name', 'Age', 'Signup_Date'], axis=1)  # Признаки
y = df['Participated_In_Promo']  # Целевая переменная

In [33]:
# Разделение данных на обучающую и валидационную выборки
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

## Нейронная сеть

### Задание слоев нейронной сети и компиляция модели

In [34]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import Precision, Recall

In [35]:
!pip install tensorflow

In [36]:
# Используем Sequential API для последовательного построения слоёв
model = Sequential([
    # Входной и первый скрытый слой (64 нейрона, ReLU-активация)
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),  
    BatchNormalization(),  # Нормализация, чтобы сделать обучение более стабильным
    Dropout(0.3),  # Отключаем 30% нейронов для предотвращения переобучения

    # Второй скрытый слой (32 нейрона)
    Dense(32, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    
    # Выходной слой (1 нейрон, сигмоид-активация для бинарной классификации)
    Dense(1, activation='sigmoid')
])

C:\Users\Alexandra\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [37]:
# Оптимизатор: Adam (адаптивный градиентный спуск)
# Функция потерь: Binary Crossentropy (для бинарной классификации)
# Метрики: Accuracy, Precision, Recall

model.compile(optimizer=Adam(learning_rate=0.001),
              loss='binary_crossentropy',
              metrics=['accuracy', Precision(name='precision'), Recall(name='recall')])

In [52]:
model_mine = Sequential([
    Dense(16, activation='tanh', input_shape=(X_train.shape[1],)),
    BatchNormalization(),
    Dropout(0.05),
    Dense(1, activation='sigmoid')
])

C:\Users\Alexandra\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [53]:
model_mine.compile(optimizer=Adam(learning_rate=0.001),
              loss='binary_crossentropy',
              metrics=['accuracy', Precision(name='precision'), Recall(name='recall')])

In [54]:
learning = model_mine.fit(X_train, y_train,
                          epochs = 4,
                          batch_size = 16,
                          validation_data=(X_valid, y_valid),
                          verbose=1 
)

Epoch 1/4
25/25 ━━━━━━━━━━━━━━━━━━━━ 4s 40ms/step - accuracy: 0.7761 - loss: 0.6750 - precision: 0.0000e+00 - recall: 0.0000e+00 - val_accuracy: 0.7778 - val_loss: 0.5390 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 2/4
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7761 - loss: 0.6390 - precision: 0.0000e+00 - recall: 0.0000e+00 - val_accuracy: 0.7778 - val_loss: 0.5412 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 3/4
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7761 - loss: 0.6054 - precision: 0.0000e+00 - recall: 0.0000e+00 - val_accuracy: 0.7778 - val_loss: 0.5447 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 4/4
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7761 - loss: 0.5794 - precision: 0.0000e+00 - recall: 0.0000e+00 - val_accuracy: 0.7778 - val_loss: 0.5497 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00


In [59]:
y_pred = model.predict(X_valid)
y_pred = (y_pred > 0.5).astype(int).flatten()

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step 


In [60]:
accuracy = accuracy_score(y_valid, y_pred)
precision = precision_score(y_valid, y_pred)
recall = recall_score(y_valid, y_pred)
f1 = f1_score(y_valid, y_pred)

print(f"Accuracy: {accuracy:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1-score: {f1:.4f}")

Accuracy: 0.7475, Precision: 0.0000, Recall: 0.0000, F1-score: 0.0000


### Обучение и оценка модели

In [57]:
history = model.fit(X_train, y_train,  # Данные для обучения
                    epochs=4,  # Количество эпох (сколько раз сеть пройдёт весь датасет)
                    batch_size=32,  # Размер батча (количество примеров за один шаг)
                    validation_data=(X_valid, y_valid),  # Проверка на валидационных данных
                    verbose=1)  # Вывод прогресса обучения

Epoch 1/4
13/13 ━━━━━━━━━━━━━━━━━━━━ 5s 73ms/step - accuracy: 0.5038 - loss: 0.8666 - precision: 0.2312 - recall: 0.5227 - val_accuracy: 0.2323 - val_loss: 2.4673 - val_precision: 0.2188 - val_recall: 0.9545
Epoch 2/4
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.4962 - loss: 0.8653 - precision: 0.1944 - recall: 0.3977 - val_accuracy: 0.3030 - val_loss: 0.8639 - val_precision: 0.2360 - val_recall: 0.9545
Epoch 3/4
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 44ms/step - accuracy: 0.5878 - loss: 0.7681 - precision: 0.2533 - recall: 0.4318 - val_accuracy: 0.5354 - val_loss: 0.6915 - val_precision: 0.2500 - val_recall: 0.5455
Epoch 4/4
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.6005 - loss: 0.7564 - precision: 0.2444 - recall: 0.3750 - val_accuracy: 0.7475 - val_loss: 0.6219 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00


In [ ]:
# Делаем предсказания на тестовой выборке
y_pred = model.predict(X_valid)
y_pred = (y_pred > 0.5).astype(int).flatten()  # Преобразуем вероятности в 0 или 1

In [ ]:
# Вычисляем метрики
accuracy = accuracy_score(y_valid, y_pred)
precision = precision_score(y_valid, y_pred)
recall = recall_score(y_valid, y_pred)
f1 = f1_score(y_valid, y_pred)

# Вывод результатов
print(f"Accuracy: {accuracy:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1-score: {f1:.4f}")